# Split Bill Business Logic

## Setup

In [1]:
# Import Libraries

import ollama
import json
import re
import time
from rapidfuzz import fuzz

### System Prompt

In [2]:
SYSTEM_PROMPT = """
You are a receipt parser. Extract only the relevant information from the receipt image.

ITEMS:
- An item is anything that was ordered (food, drinks, or services) with a price
- If the same item appears multiple times as separate lines, list each one separately
- Include all items regardless of whether the name appears abbreviated or unusual — extract exactly as written on the receipt
- If a line has no price and is at the same indentation level as the previous item, consolidate it into the previous item's name
- If a line has no price and is indented beneath the previous item, it is a sub-item description — ignore it

CHARGES:
- A charge is any financial adjustment such as tax, service charge, or discount with an explicit amount listed next to it
- Any line with an explicit amount positioned between Sub Total and Total is a charge — include it even if it appears near summary lines
- Only include a charge if it has an explicit amount listed next to it — ignore any text that merely mentions a charge within a sentence

IGNORE:
- Summary lines: Sub Total, Total, Grand Total, Cash, Change, Taxable Amount, Taxable Amt, Before Rounding, Payment
- Restaurant name, address, taglines, cashier info, loyalty points, table numbers
- Any text without a clear standalone price

Return ONLY a valid JSON object with no explanation or markdown. Use this exact schema:
{
  "items": [
    {"name": "item name", "quantity": 1, "price": 0.00}
  ],
  "charges": [
    {"name": "charge name", "amount": 0.00}
  ],
  "currency": "IDR"
}

Example:
Input: A receipt with nasi goreng, es teh, and 11% tax
Output:
{
  "items": [
    {"name": "Nasi Goreng", "quantity": 1, "price": 45000.00},
    {"name": "Es Teh", "quantity": 2, "price": 16000.00}
  ],
  "charges": [
    {"name": "PPN 11%", "amount": 6710.00}
  ],
  "currency": "IDR"
}
"""

### Helper Functions

In [3]:
import base64
from io import BytesIO
from PIL import Image

def resize_image(image_path: str, max_size: int = 1024) -> str:
    """Resize image and return as base64 string."""
    img = Image.open(image_path)
    img.thumbnail((max_size, max_size))
    buffer = BytesIO()
    img.save(buffer, format=img.format or "JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

def clean_json_output(raw: str) -> dict:
    """Strip markdown fences, remove thousands separators, and parse JSON."""
    # Strip markdown fences
    cleaned = re.sub(r"```json|```", "", raw).strip()
    # Remove thousands separators in numbers e.g. 9,637 → 9637
    cleaned = re.sub(r'(\d),(\d{3})', r'\1\2', cleaned)
    return json.loads(cleaned)

def parse_receipt(image_input: str, model: str) -> dict:
    """Send receipt image to VLM and return parsed JSON."""
    try:
        image_input = resize_image(image_input)  # 👈 resize + convert to base64
        
        response = ollama.chat(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": "Here is the receipt.",
                    "images": [image_input]
                }
            ]
        )
        raw = response["message"]["content"]
        return clean_json_output(raw)

    except json.JSONDecodeError:
        return {"error": "Model returned invalid JSON", "raw": raw}
    except Exception as e:
        return {"error": str(e)}

In [4]:
def score_results(predicted: dict, ground_truth: dict) -> dict:
    """Compare predicted output against ground truth"""
    # --- Score items ---
    name_scores = []
    price_matches = []

    for exp in ground_truth["items"]:
        best_name_score = 0
        best_price_match = False

        for pred in predicted.get("items", []):
            name_score = fuzz.ratio(
                pred["name"].lower(),
                exp["name"].lower()
            ) / 100
            if name_score > best_name_score:
                best_name_score = name_score
                best_price_match = (pred["price"] == exp["price"])

        name_scores.append(best_name_score)
        price_matches.append(best_price_match)

    # --- Score charges ---
    charge_name_scores = []
    charge_amount_matches = []

    for exp in ground_truth["charges"]:
        best_name_score = 0
        best_amount_match = False

        for pred in predicted.get("charges", []):
            name_score = fuzz.ratio(
                pred["name"].lower(),
                exp["name"].lower()
            ) / 100
            if name_score > best_name_score:
                best_name_score = name_score
                best_amount_match = (pred["amount"] == exp["amount"])

        charge_name_scores.append(best_name_score)
        charge_amount_matches.append(best_amount_match)

    return {
        "avg_item_name_score": sum(name_scores) / len(name_scores) if name_scores else 0,
        "item_price_accuracy": sum(price_matches) / len(price_matches) if price_matches else 0,
        "avg_charge_name_score": sum(charge_name_scores) / len(charge_name_scores) if charge_name_scores else 1.0,
        "charge_amount_accuracy": sum(charge_amount_matches) / len(charge_amount_matches) if charge_amount_matches else 1.0,
    }

## Parse Receipt

In [7]:
image_path = "../data/test/bon-test.jpg"

# Parse receipt
predicted = parse_receipt(image_path, model="qwen2.5vl:3b")

# Filter out zero-price items (e.g. JCO sub-items)
predicted["items"] = [i for i in predicted["items"] if i["price"] > 0]

# Display extracted items
print("EXTRACTED ITEMS:")
print(f"{'No.':<5} {'Item':<30} {'Qty':>5} {'Price':>12}")
print("-" * 55)
for idx, item in enumerate(predicted["items"], 1):
    print(f"{idx:<5} {item['name']:<30} {item['quantity']:>5} {item['price']:>12,.0f}")

# Display charges
print(f"\nCHARGES:")
print(f"{'No.':<5} {'Charge':<30} {'Amount':>12}")
print("-" * 50)
for idx, charge in enumerate(predicted["charges"], 1):
    print(f"{idx:<5} {charge['name']:<30} {charge['amount']:>12,.0f}")

EXTRACTED ITEMS:
No.   Item                             Qty        Price
-------------------------------------------------------
1     Nasi Goreng Seafood                2       89,092
2     Es Jeruk                           1       19,091
3     Es Teh Manis                       1        9,091

CHARGES:
No.   Charge                               Amount
--------------------------------------------------
1     Pb1 10%                              11,727


## Item Claim

In [8]:
# Define People and Claims
# Key: person name
# Value: dict of {item_name: claimed_quantity}

claims = {
    "Alice": {
        "Nasi Goreng Seafood": 1,
        "Es Jeruk": 1
    },
    "Bob": {
        "Nasi Goreng Seafood": 1,
        "Es Teh Manis": 1
    }
}

# Display claims
print("CLAIMS:")
print(f"{'Person':<15} {'Item':<30} {'Qty':>5}")
print("-" * 52)
for person, items in claims.items():
    for item_name, qty in items.items():
        print(f"{person:<15} {item_name:<30} {qty:>5}")

CLAIMS:
Person          Item                             Qty
----------------------------------------------------
Alice           Nasi Goreng Seafood                1
Alice           Es Jeruk                           1
Bob             Nasi Goreng Seafood                1
Bob             Es Teh Manis                       1


In [9]:
# Validate Claims

def validate_claims(claims: dict, items: list) -> bool:
    # Build item lookup dict
    item_lookup = {item["name"]: item for item in items}
    
    # Sum claimed quantities per item across all people
    claimed_totals = {}
    for person, claimed_items in claims.items():
        for item_name, qty in claimed_items.items():
            if item_name not in claimed_totals:
                claimed_totals[item_name] = 0
            claimed_totals[item_name] += qty
    
    # Validate
    is_valid = True
    for item_name, total_claimed in claimed_totals.items():
        # Check item exists in receipt
        if item_name not in item_lookup:
            print(f"❌ '{item_name}' tidak ditemukan di struk")
            is_valid = False
            continue
        
        # Check claimed quantity doesn't exceed available
        available = item_lookup[item_name]["quantity"]
        if total_claimed > available:
            print(f"❌ '{item_name}': total klaim {total_claimed} melebihi tersedia {available}")
            is_valid = False
    
    if is_valid:
        print("✅ Semua klaim valid!")
    
    return is_valid

# Run validation
is_valid = validate_claims(claims, predicted["items"])

✅ Semua klaim valid!


## Calculate Subtotals

In [10]:
def calculate_subtotals(claims: dict, items: list) -> dict:
    # Convert items list to a lookup dict for easy access
    item_lookup = {item["name"]: item for item in items}
    
    subtotals = {}
    for person, claimed_items in claims.items():
        subtotal = 0
        for item_name, claimed_qty in claimed_items.items():
            item = item_lookup[item_name]
            unit_price = item["price"] / item["quantity"]
            subtotal += unit_price * claimed_qty
        subtotals[person] = subtotal
    
    return subtotals

In [14]:
if is_valid:
    subtotals = calculate_subtotals(claims, predicted["items"])
    
    print("SUBTOTALS:")
    print(f"{'Person':<15} {'Subtotal':>12}")
    print("-" * 30)
    for person, subtotal in subtotals.items():
        print(f"{person:<15} {subtotal:>12,.0f}")
else:
    print("⚠️ Perbaiki klaim sebelum melanjutkan")

SUBTOTALS:
Person              Subtotal
------------------------------
Alice                 63,637
Bob                   53,637


## Calculate Charges

In [15]:
def calculate_charges(charges: list, subtotals: dict):
    total_subtotal = 0
    for key, value in subtotals.items():
        total_subtotal += value

    splitted_charges = {person: 0 for person in subtotals}
    
    for charge in charges:
        for person, total in subtotals.items():    
            splitted_charges[person] += total / total_subtotal * charge["amount"]
        
    return splitted_charges

In [16]:
if is_valid:
    charges = calculate_charges(predicted["charges"], subtotals)
    
    print("Charges:")
    print(f"{'Person':<15} {'Charges':>12}")
    print("-" * 30)
    for person, charge in charges.items():
        print(f"{person:<15} {charge:>12,.0f}")
else:
    print("⚠️ Perbaiki klaim sebelum melanjutkan")

Charges:
Person               Charges
------------------------------
Alice                  6,363
Bob                    5,364


## Return Splitted Bill

In [17]:
if is_valid:
    print("Totals:")
    print(f"{'Person':<15} {'Totals':>12}")
    print("-" * 30)
    for person in claims:
        total = subtotals[person] + charges[person]
        print(f"{person:<15} {total:>12,.0f}")
else:
    print("⚠️ Perbaiki klaim sebelum melanjutkan")

Totals:
Person                Totals
------------------------------
Alice                 70,000
Bob                   59,001
